# EvalRef3D v1.0 — Model Evaluation Certificate
Use **Runtime → Change runtime type → CPU**, then **Run all**. No GPU or T4 is required.
Upload one EvalRef3D Request ZIP created in the browser. The request includes Ground Truth and raw prediction labelmaps, not source images or model weights. Labelmaps and metadata may still be identifiable.
The output is an unsigned, self-evaluated research record bound to the original Model ZIP SHA-256. It does not guarantee performance or establish clinical validity. The case count and evaluated data domain must accompany any reported score.


In [ ]:
from pathlib import Path
from google.colab import files
import io
import zipfile

REUPLOAD_INPUTS = False
request_path = Path('/content/evalref3d_request.zip')
if REUPLOAD_INPUTS or not request_path.is_file():
    uploaded = files.upload()
    try:
        assert len(uploaded) == 1, 'Upload exactly one EvalRef3D Request ZIP.'
        name, data = next(iter(uploaded.items()))
        assert name.lower().endswith('.zip') and len(data) <= 1024**3, 'Expected a ZIP up to 1 GiB.'
        assert zipfile.is_zipfile(io.BytesIO(data)), 'Invalid ZIP.'
        request_path.write_bytes(data)
    finally:
        del uploaded
else:
    print('Reusing saved Evaluation Request ZIP. Set REUPLOAD_INPUTS=True to replace it.')


## Set up the CPU evaluation runtime
The backend and both shared validators are downloaded from the same repository revision. Pin BACKEND_REF to a reviewed commit for archival reproduction. All three source hashes and actual package versions are recorded in the certificate.


In [ ]:
BACKEND_REF = 'main'  # Replace with a reviewed commit SHA for a reproducible deployment.
import subprocess
import sys
import urllib.request

backend_dir = Path('/content/evalref3d_backend')
backend_dir.mkdir(exist_ok=True)
for filename in ('evalref3d_backend.py', 'inferref3d_backend.py', 'trainref3d_backend.py'):
    url = f'https://raw.githubusercontent.com/SatoruMuro/SegRef3D/{BACKEND_REF}/ColabNotebooks/{filename}'
    urllib.request.urlretrieve(url, backend_dir / filename)
# Run evaluation in a fresh subprocess, so Colab's pre-imported numpy cannot mask these pins.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       'numpy==2.2.6', 'nibabel==5.3.2', 'scipy==1.15.3'])


## Validate, calculate metrics and download
No resampling or postprocessing is performed. Invalid identity, geometry or overlap is rejected.


In [ ]:
import json
import tempfile

output_dir = Path(tempfile.mkdtemp(prefix='evalref3d_output_', dir='/content'))
subprocess.check_call([sys.executable, str(backend_dir / 'evalref3d_backend.py'),
                       str(request_path), '--output-dir', str(output_dir)])
outputs = list(output_dir.glob('TrainRef3D_Evaluation_TR3DE_*.zip'))
assert len(outputs) == 1, 'Expected one certificate.'
with zipfile.ZipFile(outputs[0]) as archive:
    certificate = json.loads(archive.read('evaluation_certificate.json'))
print(f"Self-evaluated independent holdout: n={certificate['evaluation_set']['case_count']}")
print('Domain:', certificate['evaluation_set']['domain'])
print('Dice summary:', certificate['summary']['dice'])
print('Raw prediction only. Not clinically validated.')
files.download(str(outputs[0]))
